# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset, which details clinical and molecular features of second primary colorectal cancer in cancer survivors, using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and provided via this URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`.

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
We load the dataset's metadata via the provided Croissant schema using `mlcroissant`. This step also displays basic dataset information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"\u001b[1m{metadata.name}\u001b[0m")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"License: {metadata.license}")

## 2. Data Overview
Explore the available record sets and their contents using their `@id` fields.

> **Tip:** In Croissant datasets, `record_sets`, `fields`, and `columns` are referred to by their unique `@id` values for programmatic access.

In [ ]:
# Discover available record sets and their fields by @id
record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]

print("Available RecordSet @id values:")
if not record_sets:
    # If no record sets listed directly, try auto-discover
    from mlcroissant.dataset.metadata import _find_record_sets
    record_sets = [r['@id'] for r in _find_record_sets(dataset.metadata, dataset.metadata.to_json())]
for rsid in record_sets:
    print(f"  - {rsid}")

# Show fields for each record set
for rsid in record_sets:
    print(f"\nFields for RecordSet {rsid} :")
    # Each record set in metadata has 'field' or 'fields' with @id's
    rs_obj = None
    # Try to fetch record set object by id
    for rs in dataset.metadata.to_json().get('recordSet', []):
        if rs['@id'] == rsid:
            rs_obj = rs
            break
    if rs_obj is None:
        # Use helper to get record set dict if not listed above
        for rs in _find_record_sets(dataset.metadata, dataset.metadata.to_json()):
            if rs['@id'] == rsid:
                rs_obj = rs
                break
    if rs_obj:
        fields = rs_obj.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                field_id = field.get('@id')
                field_name = field.get('name', '')
            else:
                field_id = field
                field_name = ''
            print(f"    - {field_id} {f'({field_name})' if field_name else ''}")
    else:
        print("    (No field info found in metadata for this record set)")

## 3. Data Extraction
Next, we'll extract the complete data table(s) for analysis. 

We load all records for each record set using its `@id` into a pandas DataFrame.

In [ ]:
# Collect data for each record set by @id
dataframes = {}
for rsid in record_sets:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for RecordSet {rsid} with shape", df.shape)
    else:
        print(f"No records extracted for RecordSet {rsid}")

# Display column names and first five rows for the first main record set (if any)
if dataframes:
    main_record_set = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for RecordSet {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    dataframes[main_record_set].head()
else:
    print("No dataframes loaded!")

## 4. Exploratory Data Analysis (EDA)
We demonstrate filtering, normalization, and grouping on a numerical field. All field and record set references use their exact `@id` values for reproducibility.

> ⚠️ **Note:** Replace `<NUMERIC_FIELD_ID>` and `<GROUP_FIELD_ID>` if the default guesses do not match your schema. This cell auto-detects the most likely numeric field.

In [ ]:
# EDA for the main record set
record_set_id = main_record_set if dataframes else None
df = dataframes[record_set_id] if record_set_id else None

# Try to find a numeric column by checking pandas dtype
numeric_field_id = None
if df is not None:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        numeric_field_id = num_cols[0]
        print(f"Auto-selected numeric field: {numeric_field_id}")
    else:
        # Try to guess columns with age or similar
        for col in df.columns:
            if 'age' in col.lower():
                numeric_field_id = col
                print(f"Guessed numeric field: {numeric_field_id}")
                break

    # Set threshold for filtering (arbitrary: e.g., 50 for age)
    if numeric_field_id:
        # Handle missing values sensibly
        _col = df[numeric_field_id]
        threshold = 50 if 'age' in numeric_field_id.lower() else (_col.mean() if _col.dtype.kind in 'if' else 10)
        filtered_df = df[_col > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std else 0
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to find a categorical/group field
        # Prefer one with a small number of unique values
        group_field_id = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            nunique = df[col].nunique(dropna=True)
            if 2 <= nunique <= 6:
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by {group_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("Cannot perform EDA: No main record set loaded.")

## 5. Visualization
We visualize the filtered and normalized data, such as histogram for the numeric field and a boxplot by a group field if one is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric variable
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field_id].dropna(), bins=15, color='teal', alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if group_field_id exists
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We have demonstrated loading, overview, and basic exploratory analysis of the FAIR² clinical colorectal cancer dataset using `mlcroissant`.

- Data was referenced and accessed using Croissant `@id`s for robust, reproducible analysis.
- Both metadata and tabular data from all available record sets were loaded and previewed.
- Example processing: numeric field filtering, normalization, grouping, and visualization provide a template for further clinical analysis.

For detailed modeling or publication, please review the dataset's licensing (`https://opendatacommons.org/licenses/by/1-0/`) and cite as instructed in the dataset metadata.